# Basic Modules


In [1]:
import numpy as np

## DropOut
Randomly set the elements to zero based on given probability. 

In [ ]:
class DropOut:
    def __init__(self, p:float=0.1):
        self.p = p
    
    def forward(self, x:np.ndarray, train:bool=True) -> np.ndarray:
        
        if train:
            scale = 1. / (1. - self.p)
            self.mask = np.random.rand(*x.shape) > self.p
            return x * self.mask * scale
        else:
            return x
    
    def backward(self, dout:np.ndarray) -> np.ndarray:
        return dout * self.mask
    


## Batch Norm


In [3]:
class BatchNorm:
    def __init__(self, num_features: int, eps: float = 1e-5, momentum: float = 0.1):
        self.num_features = num_features
        self.eps = eps
        self.momentum = momentum

        # learnable parameters
        self.gamma = np.ones(num_features)   # scale
        self.beta = np.zeros(num_features)   # shift

        # running statistics for inference
        self.running_mean = np.zeros(num_features)
        self.running_var = np.ones(num_features)

        # gradients for learnable parameters
        self.dgamma = None
        self.dbeta = None

    def forward(self, x: np.ndarray, train: bool = True) -> np.ndarray:
        """
        x: (N, D) where D == num_features
        """
        if train:
            mean = x.mean(axis=0)                          # (D,)
            var = x.var(axis=0)                            # (D,)

            # update running statistics
            self.running_mean = (1 - self.momentum) * self.running_mean + self.momentum * mean
            self.running_var = (1 - self.momentum) * self.running_var + self.momentum * var

            # normalize
            self.x_hat = (x - mean) / np.sqrt(var + self.eps)   # (N, D)
        else:
            self.x_hat = (x - self.running_mean) / np.sqrt(self.running_var + self.eps)

        # cache for backward
        self.x = x
        self.mean = mean if train else self.running_mean
        self.var = var if train else self.running_var
        self.N = x.shape[0]

        out = self.gamma * self.x_hat + self.beta          # (N, D)
        return out

    def backward(self, dout: np.ndarray) -> np.ndarray:
        """
        dout: (N, D) gradient from upstream
        returns: dx (N, D)
        """
        N = self.N

        # gradients of learnable parameters
        self.dgamma = np.sum(dout * self.x_hat, axis=0)   # (D,)
        self.dbeta = np.sum(dout, axis=0)                  # (D,)

        # gradient through normalization
        dx_hat = dout * self.gamma                         # (N, D)
        std_inv = 1.0 / np.sqrt(self.var + self.eps)      # (D,)

        dx = (1.0 / N) * std_inv * (
            N * dx_hat
            - np.sum(dx_hat, axis=0)
            - self.x_hat * np.sum(dx_hat * self.x_hat, axis=0)
        )
        return dx


## LayerNorm

Layer norm is 

In [4]:
class LayerNorm:
    def __init__(self, normalized_shape, eps: float = 1e-5):
        """
        normalized_shape: int or tuple of ints for the last dimension(s) to normalize over.
                          e.g. (D,) for input (N, D) or (H, W) for input (N, H, W)
        """
        if isinstance(normalized_shape, int):
            normalized_shape = (normalized_shape,)
        self.normalized_shape = tuple(normalized_shape)
        self.eps = eps

        # learnable parameters, same shape as normalized dimensions
        self.gamma = np.ones(self.normalized_shape)
        self.beta = np.zeros(self.normalized_shape)

        # gradients
        self.dgamma = None
        self.dbeta = None

    def forward(self, x: np.ndarray) -> np.ndarray:
        """
        x: arbitrary shape (..., *normalized_shape)
        Normalizes over the last len(normalized_shape) dimensions.
        """
        self.x = x
        # axes to reduce over: the last len(normalized_shape) dims
        ndim = len(self.normalized_shape)
        self.norm_axes = tuple(range(-ndim, 0))

        self.mean = x.mean(axis=self.norm_axes, keepdims=True)
        self.var = x.var(axis=self.norm_axes, keepdims=True)
        self.x_hat = (x - self.mean) / np.sqrt(self.var + self.eps)

        out = self.gamma * self.x_hat + self.beta
        return out

    def backward(self, dout: np.ndarray) -> np.ndarray:
        """
        dout: same shape as x
        returns: dx, same shape as x
        """
        # number of elements in the normalized dimensions
        n = 1
        for s in self.normalized_shape:
            n *= s

        # gradients of learnable parameters: sum over batch (all non-normalized dims)
        batch_axes = tuple(range(len(dout.shape) - len(self.normalized_shape)))
        self.dgamma = np.sum(dout * self.x_hat, axis=batch_axes)
        self.dbeta = np.sum(dout, axis=batch_axes)

        # gradient through normalization
        dx_hat = dout * self.gamma
        std_inv = 1.0 / np.sqrt(self.var + self.eps)

        dx = (1.0 / n) * std_inv * (
            n * dx_hat
            - dx_hat.sum(axis=self.norm_axes, keepdims=True)
            - self.x_hat * (dx_hat * self.x_hat).sum(axis=self.norm_axes, keepdims=True)
        )
        return dx


In [5]:
x = np.random.randn(4, 5)  # example input

x.mean(axis=-1).shape

(4,)